In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from collections import deque
from collections import defaultdict
import os
from tqdm import tqdm
from utlis import *
%load_ext autoreload
%autoreload 2

# Download wikipedia data

In [ ]:
# bash /cephfs/liuxinyu/wikiKG/scripts/download_dumps.sh

# Extracting category links from raw wikipedia data

In [ ]:
# parses Wikipedia SQL dump files to extract the category graph for a knowledge graph
# python -m wikikg.category.parse_categories \
#   --page-sql data/raw/page.sql.gz \
#   --category-sql data/raw/category.sql.gz \
#   --linktarget-sql data/raw/linktarget.sql.gz \
#   --categorylinks-sql data/raw/categorylinks.sql.gz \
#   --out-dir data/graph/category \
#   --workers 8

# Extract semantic subgraph from main topic classification

In [ ]:
# filters those raw files down to a clean semantic category subgraph in 5 steps:

# 1. Pattern filter — removes non-semantic categories matching patterns like Wikipedia_*, *_stubs, List_of_*, *_by_country, *_people, etc. 
# (administrative, maintenance, list-type categories)

# 2. BFS from root — starting at Main_topic_classifications, does a breadth-first traversal of the parent→child hierarchy and 
# keeps only categories reachable from that root

# 3. Remove isolated leaves (iterative) — repeatedly prunes leaf categories that have fewer than min_pages_for_leaf=5 articles, 
# until no more can be removed (cascading: removing a leaf can expose new leaves)

# 4. Filter page-category links — keeps only (page_id, category_id) rows where the category survived steps 1–3

# 5. Save to /cephfs/liuxinyu/wikiKG/tests/semantic_category/:

# semantic_categories.parquet
# semantic_category_edges.parquet
# semantic_page_categories.parquet

# Return value: reachable = the set of category_ids that made it into the final subgraph.

reachable = extract_semantic_subgraph(
    "/cephfs/liuxinyu/wikiKG/data/graph/category",
    "/cephfs/liuxinyu/wikiKG/tests/semantic_category",
    root_title="Main_topic_classifications",
    apply_pattern_filter=True,
    remove_isolated=True,
    min_pages_for_leaf=5,
)

Loading categories...
  Loaded 2597047 categories
Loading edges...
  Loaded 9779068 edges
Loading page-category relationships...
  Loaded 97424611 page-category links

[Step 1] Applying pattern-based filtering...
  Removed 280805 non-semantic categories
  Remaining: 2316242 categories
  Removed 2616677 edges

[Step 2] Finding categories reachable from root...
  Root: 'Main_topic_classifications' (id: 192834)


  BFS traversal: 2015840 nodes [00:02, 721135.46 nodes/s]


  Found 2015841 reachable categories

[Step 3] Removing isolated leaf categories (min_pages=5)...
  Iteration 1: Removing 462981 isolated categories
  Iteration 2: Removing 136559 isolated categories
  Iteration 3: Removing 39282 isolated categories
  Iteration 4: Removing 10550 isolated categories
  Iteration 5: Removing 3069 isolated categories
  Iteration 6: Removing 1018 isolated categories
  Iteration 7: Removing 206 isolated categories
  Iteration 8: Removing 44 isolated categories
  Iteration 9: Removing 7 isolated categories
  Iteration 10: Removing 4 isolated categories
  Iteration 11: Removing 3 isolated categories
  Iteration 12: Removing 1 isolated categories
  Iteration 13: No more isolated categories to remove
  Final count after isolation removal: 1362117 categories

[Step 4] Filtering page-category links...

FINAL SUMMARY
  Categories: 1362117
  Edges: 3284150
  Page-category links: 37584659
  Leaf categories: 614890
  Internal categories: 747227

[Step 5] Saving filter

# Semantic pruning

In [2]:
config = SemanticTreeConfig(
    k_parents=1,
    root_title="Main_topic_classifications",
)
input_dir = "/cephfs/liuxinyu/wikiKG/tests/semantic_category"
output_dir = "/cephfs/liuxinyu/wikiKG/tests/semantic_category_tree"
builder = SemanticTreeBuilder(config)
builder.build(input_dir=input_dir,output_dir=output_dir)

Building Semantic Tree (k=1 parents)
Loading data...
  Categories: 1362117
  Edges: 3284150
  Page-category links: 37584659
Building lookups...
  Root: 'Main_topic_classifications' (id: 192834)
  Anchor categories found: 38

Pruning to top-1 semantic parents per node...
Computing depths from root...
  Computed depths for 1362117 nodes
  Max depth: 19


  Selecting best paths: 100%|██████████| 1362116/1362116 [01:14<00:00, 18222.95it/s]


  Extracting parent choices...

  Selection statistics:
    Single parent (no choice): 641793
    Avoided penalized paths: 101166
    Preferred domain paths: 30366
    Preferred anchor paths: 92807
    Depth tiebreaker: 178864
  Pruned edges: 1362116

Verifying connectivity from root...
  Reachable: 1362117
  Removed: 0 categories

Computing final depths...
  Max depth: 20

FINAL SUMMARY
  Categories: 1362117
  Edges: 1362116
  Page-category links: 37584659
  Max depth: 20
  Depth distribution:
    Depth 0: 1 categories
    Depth 1: 33 categories
    Depth 2: 797 categories
    Depth 3: 6518 categories
    Depth 4: 30211 categories
    Depth 5: 91245 categories
    Depth 6: 181347 categories
    Depth 7: 273511 categories
    Depth 8: 307969 categories
    Depth 9: 258678 categories
    Depth 10: 125649 categories
    Depth 11: 51678 categories
    Depth 12: 22069 categories
    Depth 13: 7337 categories
    Depth 14: 3293 categories
    ... (21 total depth levels)
  Parents per catego

In [67]:
index = CategoryIndex("/cephfs/liuxinyu/wikiKG/tests/sembantic_category_tree")

Loading data from /cephfs/liuxinyu/wikiKG/tests/semantic_category_tree...
Building index...
Index ready: 1362117 categories, 1362116 edges


In [4]:
index.query("Nature", depth=10)
index.query("Science", depth=10)
index.query("Culture", depth=10)
index.query("Society", depth=10)
index.query("Technology", depth=10)

# -----------------------------------------------------------------------------
# MID-LEVEL CONCEPTS (Depth 3-6)
# These should show clear domain routing
# -----------------------------------------------------------------------------

# Life sciences branch
index.query("Animals", depth=10)      # Should go through Life/Nature
index.query("Mammals", depth=10)      # Should go through Animals
index.query("Plants", depth=10)       # Should go through Life/Nature

# Physical sciences branch  
index.query("Physics", depth=10)      # Should go through Science
index.query("Chemistry", depth=10)    # Should go through Science
index.query("Mathematics", depth=10)  # Should go through Science or direct

# Human domains
index.query("Arts", depth=10)         # Should go through Culture
index.query("Music", depth=10)        # Should go through Arts
index.query("Literature", depth=10)   # Should go through Arts or Culture
index.query("Sports", depth=10)       # Should go through Culture or Activities

# Geography/Places
index.query("Countries", depth=10)    # Should go through Geography or Places

# -----------------------------------------------------------------------------
# LOW-LEVEL CONCEPTS (Depth 7+)
# These are the critical tests - should NOT go through weird paths
# -----------------------------------------------------------------------------

# Animals - should go through Animals, NOT through Culture/Ethnobiology
index.query("Dogs", depth=10)
index.query("Cats", depth=10)
index.query("Horses", depth=10)
index.query("Elephants", depth=10)
index.query("Eagles", depth=10)       # Bird - should go through Animals

# Plants - should go through Plants, NOT through Culture/Agriculture
index.query("Roses", depth=10)
index.query("Oak", depth=10)          # Tree

# Specific sciences
index.query("Quantum_mechanics", depth=10)    # Should go through Physics
index.query("Organic_chemistry", depth=10)    # Should go through Chemistry
index.query("Genetics", depth=10)             # Should go through Biology

# Cultural items - these SHOULD go through Culture
index.query("Jazz", depth=10)                 # Music genre
index.query("Impressionism", depth=10)        # Art movement
index.query("Poetry", depth=10)               # Literature

# Countries - should go through Geography/Places
index.query("France", depth=10)
index.query("Japan", depth=10)
index.query("Brazil", depth=10)

# Technology
index.query("Programming_languages", depth=10)
index.query("Artificial_intelligence", depth=10)
index.query("Internet", depth=10)

# -----------------------------------------------------------------------------
# EDGE CASES & POTENTIAL PROBLEM AREAS
# These are categories that might have multiple valid parents
# -----------------------------------------------------------------------------

# Domesticated animals - the original problem case
index.query("Domesticated_animals", depth=10)  # Should go to Animals, NOT Ethnobiology
index.query("Pets", depth=10)                  # Should go to Animals
index.query("Livestock", depth=10)             # Could be Animals or Agriculture

# Food-related (ambiguous - could be Culture or Biology)
index.query("Fruit", depth=10)                 # Should probably go through Plants
index.query("Vegetables", depth=10)            # Should probably go through Plants
index.query("Meat", depth=10)                  # Could go through Food or Animals

# Human roles (ambiguous)
index.query("Scientists", depth=10)            # People? Science?
index.query("Musicians", depth=10)             # People? Music?
index.query("Athletes", depth=10)              # People? Sports?

# Applied/interdisciplinary
index.query("Medicine", depth=10)              # Science? Health?
index.query("Agriculture", depth=10)           # Technology? Biology?
index.query("Architecture", depth=10)          # Arts? Technology?



CATEGORY: Nature (ID: 215694)
Direct pages: 38

[UP TREE]
Nature
  └── Main_topic_classifications

CATEGORY: Science (ID: 275726)
Direct pages: 39

[UP TREE]
Science
  └── Main_topic_classifications

CATEGORY: Culture (ID: 103681)
Direct pages: 99

[UP TREE]
Culture
  └── Main_topic_classifications

CATEGORY: Society (ID: 284860)
Direct pages: 76

[UP TREE]
Society
  └── Main_topic_classifications

CATEGORY: Technology (ID: 300026)
Direct pages: 93

[UP TREE]
Technology
  └── Main_topic_classifications

CATEGORY: Animals (ID: 49634)
Direct pages: 53

[UP TREE]
Animals
  └── Organisms
    └── Life
      └── Nature
        └── Main_topic_classifications

CATEGORY: Mammals (ID: 193859)
Direct pages: 33

[UP TREE]
Mammals
  └── Animals
    └── Organisms
      └── Life
        └── Nature
          └── Main_topic_classifications

CATEGORY: Plants (ID: 248124)
Direct pages: 88

[UP TREE]
Plants
  └── Organisms
    └── Life
      └── Nature
        └── Main_topic_classifications

CATEGORY: Ph

In [26]:
index.query("Lakes", depth=10)          # Arts? Technology?


CATEGORY: Lakes (ID: 180904)
Direct pages: 55

[UP TREE]
Lakes
  └── Landscape
    └── Geography
      └── Main_topic_classifications


# Attach nodes

In [ ]:
# Generate nodes_visual

# hyperlink pipeline
# python -m wikikg.hyperlink.parse_dumps \
#     --page-sql data/raw/page.sql.gz \
#     --linktarget-sql data/raw/linktarget.sql.gz \
#     --pagelinks-sql data/raw/pagelinks.sql.gz \
#     --redirect-sql data/raw/redirect.sql.gz \
#     --out-dir data/graph/hyperlink \
#     --workers 7

# python wikikg/hyperlink/compute_pagerank.py \ 
#     --nodes data/graph/hyperlink/nodes.parquet \
#     --edges data/graph/hyperlink/edges.parquet \
#     --out data/graph/hyperlink/pagerank.parquet \
#     --stats \
#     --exclude-titles /cephfs/liuxinyu/wikiKG/data/graph/hyperlink/exclude_titles.txt

# python wikikg/hyperlink/query_pagerank.py --nodes data/graph/hyperlink/nodes.parquet --pagerank data/graph/hyperlink/pagerank.parquet --title Dog

# python wikikg/hyperlink/filter_by_pagerank.py \
#     --nodes data/graph/hyperlink/nodes.parquet \
#     --edges data/graph/hyperlink/edges.parquet \
#     --pagerank data/graph/hyperlink/pagerank.parquet \
#     --out-nodes data/graph/hyperlink/nodes_filtered.parquet \
#     --out-edges data/graph/hyperlink/edges_filtered.parquet \
#     --percentile 99ß

# python wikikg/hyperlink/classify_llm.py \
#     --nodes data/graph/hyperlink/nodes_filtered.parquet \
#     --out-visual data/graph/hyperlink/nodes_visual.parquet \
#     --out-nonvisual data/graph/hyperlink/nodes_nonvisual.parquet \
#     --out-failed data/graph/hyperlink/nodes_failed.parquet \
#     --model gemini-3-flash-preview \
#     --batch-size 30 \
#     --workers 16 

In [38]:
output_dir = "/cephfs/liuxinyu/wikiKG/tests/semantic_graph"
config = SemanticTreeConfig(
    k_parents=1,
    root_title="Main_topic_classifications",
)
builder = KnowledgeGraphBuilder(config)

builder.build(
    semantic_tree_dir='/cephfs/liuxinyu/wikiKG/tests/semantic_category_tree',
    visual_nodes_path="/cephfs/liuxinyu/wikiKG/data/graph/hyperlink/nodes_visual.parquet",
    page_categories_path="/cephfs/liuxinyu/wikiKG/tests/semantic_category_tree/semantic_page_categories.parquet",
    output_dir=output_dir,
)

LOADING SEMANTIC TREE
Loading categories from /cephfs/liuxinyu/wikiKG/tests/semantic_category_tree/semantic_categories.parquet...
  1362117 categories
Loading edges from /cephfs/liuxinyu/wikiKG/tests/semantic_category_tree/semantic_category_edges.parquet...
  1362116 edges
Building lookups...
  Root: 'Main_topic_classifications' (id: cat_192834)
Computing depths...
  Max depth: 20
Computing paths to root...
  Found 38 anchor categories

Loading visual nodes from /cephfs/liuxinyu/wikiKG/data/graph/hyperlink/nodes_visual.parquet...
  91897 visual pages
  Columns: ['page_id', 'title']

Loading page categories from /cephfs/liuxinyu/wikiKG/tests/semantic_category_tree/semantic_page_categories.parquet...
  37584659 page-category links

BUILDING UNIFIED KNOWLEDGE GRAPH
  Visual pages to attach: 91897
  Category titles for dedup: 1362087
  Building page-category mappings...


  Filtering: 100%|██████████| 37584659/37584659 [00:21<00:00, 1726015.75it/s]


  Pages with valid category mappings: 91643
  Pages without valid categories: 254

  Selecting best category for each page...


  Scoring: 100%|██████████| 91643/91643 [00:15<00:00, 5854.71it/s]



  Page attachment statistics:
    Single category (no choice): 4049
    Avoided penalized paths: 2300
    Preferred domain paths: 1223
    Preferred anchor paths: 15376
    Preferred deeper category: 0
    Length tiebreaker: 35341
  Skipped:
    Same name as category (attached to it): 32219
    Same name as category (couldn't attach): 1135
    No valid category: 0

  Building unified node table...
  Building unified edge table...
  Pruning empty categories...
    Removing 955909 empty leaf categories...
    Removing 222226 empty leaf categories...
    Removing 64723 empty leaf categories...
    Removing 18923 empty leaf categories...
    Removing 5219 empty leaf categories...
    Removing 1476 empty leaf categories...
    Removing 375 empty leaf categories...
    Removing 54 empty leaf categories...
    Removing 12 empty leaf categories...
    Removing 3 empty leaf categories...
    Removing 1 empty leaf categories...

FINAL SUMMARY

Unified Knowledge Graph:
  Total nodes: 183704
    

In [39]:
del index

In [ ]:
index = KnowledgeGraphIndex("/cephfs/liuxinyu/wikiKG/tests/semantic_graph")

Loading unified knowledge graph from /cephfs/liuxinyu/wikiKG/tests/semantic_graph...
  Total Nodes: 183704
  - Categories: 93196
  - Pages: 90508
  Ready.



In [66]:
index.nodes

,node_id,title,node_type,depth,parent_id
0,cat_3,Computer_storage_devices,category,4,cat_173430763
1,cat_10,Rivers_of_Vietnam,category,9,cat_140512174
2,cat_16,Comedy,category,4,cat_122473
3,cat_20,NASCAR_teams,category,6,cat_58469
4,cat_21,Muhammad_Ali,category,5,cat_249534287
...,...,...,...,...,...
183699,page_81889433,African_Nations_League,page,11,cat_250191913
183700,page_81896973,AFC_Nations_League,page,8,cat_249290681
183701,page_81931154,1901_Nobel_Prize_in_Physics,page,11,cat_207269975
183702,page_81965930,FDR_Media,page,6,cat_262366


In [36]:
# Check what Lightning entries exist
lightning_nodes = index.nodes[index.nodes['title'].str.lower() == 'lightning']
print(lightning_nodes)

# Check the title_to_id index
print(index.title_to_id.get('lightning', []))

Empty DataFrame
Columns: [node_id, title, node_type, depth, parent_id]
Index: []
[]


In [62]:
index.query("physics")

CATEGORY: Applied and interdisciplinary physics

HIERARCHY PATH:
--------------------------------------------------
📁 Main_topic_classifications [L0]
  └── 📁 Science [L1]
    └── 📁 Branches_of_science [L2]
      └── 📁 Applied_sciences [L3]
        └── 📁 Applied_and_interdisciplinary_physics [L4]

DIRECT CHILDREN:
--------------------------------------------------
  📁 Optics
  📁 Physical_oceanography
  📁 Geophysics
  📁 Physical_chemistry

SIBLINGS (under Applied_sciences):
--------------------------------------------------
  📁 Forensic_science
  📁 Space_science
  📁 Metrology
  📁 Sports_science


CATEGORY: Atomic physics

HIERARCHY PATH:
--------------------------------------------------
📁 Main_topic_classifications [L0]
  └── 📁 Nature [L1]
    └── 📁 Matter [L2]
      └── 📁 Atoms [L3]
        └── 📁 Atomic_physics [L4]

DIRECT CHILDREN:
--------------------------------------------------
  📁 Electron
  ... and 1 pages

SIBLINGS (under Atoms):
-----------------------------------------------